# Exp5 Part A — Dry run + full test blocking

**Goal:** prove the heavy step works (dry run), then generate ALL test candidates.

**You run:** cells 0 → 1 (paste output, WAIT for go) → cells 2 → 3.
Then Part B (separate notebook) does train + inference + submission.

**If the runtime dies:** re-run from cell 0 — `git lfs pull` and every `--resume`
cell skip finished work. Files already copied to Drive survive.

**Locked config:** char (3,4), min_df=2, max_df=0.05, K=40/channel.

In [ ]:
# 0) Setup: Drive, repo (clone once, pull after), data, deps. (~15-25 min first time)
from google.colab import drive
drive.mount('/content/drive')
!if [ -d ML-challenge ]; then cd ML-challenge && git checkout arena/01a0df98-ml-challenge && git pull; else git clone -b arena/01a0df98-ml-challenge https://github.com/Saadmadni84/ML-challenge.git && cd ML-challenge; fi
%cd /content/ML-challenge
!git lfs pull
!ls -lh dataset/train dataset/test
!pip -q install -r code/business_entity_resolution/requirements.txt
!python3 -c "import sklearn, lightgbm, rapidfuzz, sparse_dot_topn; print('deps OK')"

In [ ]:
# 1) DRY RUN: 3k test queries/country against FULL targets (~20-40 min).
# Builds the full indices (the heaviest step) - that is the point. PASTE OUTPUT.
S = "code/business_entity_resolution/src"
!python3 $S/blocking.py --s1 dataset/test/test_source1.tsv --s2 dataset/test/test_source2.tsv --s3 dataset/test/test_source3.tsv --field name --out output/dry_name_{country}.tsv --k 40 --min-ngram 3 --max-ngram 4 --min-df 2 --max-df 0.05 --countries India,US,France --max-q 3000
!python3 $S/blocking.py --s1 dataset/test/test_source1.tsv --s2 dataset/test/test_source2.tsv --s3 dataset/test/test_source3.tsv --field address --out output/dry_addr_{country}.tsv --k 40 --min-ngram 3 --max-ngram 4 --min-df 2 --max-df 0.05 --countries India,US,France --max-q 3000

⏸️ **PAUSE — paste the dry-run output (index sizes, S1/s, ETAs) in chat and WAIT.**
We confirm the budget before launching the hours-long cell 2.

In [ ]:
# 2) FULL TEST BLOCKING, per country+field with --resume (hours; ETA printed).
# Run ONLY after the dry run was approved. Leave the tab open. Re-runnable.
S = "code/business_entity_resolution/src"
for C in ["India", "US", "France"]:
    for F in ["name", "address"]:
        print(f"=== {C} / {F} ===", flush=True)
        !python3 $S/blocking.py --s1 dataset/test/test_source1.tsv --s2 dataset/test/test_source2.tsv --s3 dataset/test/test_source3.tsv --field $F --out output/test_{country}_{field}.tsv --k 40 --min-ngram 3 --max-ngram 4 --min-df 2 --max-df 0.05 --countries $C --resume
        !cp output/test_{country}_{field}.tsv "/content/drive/MyDrive/test_{country}_{field}.tsv"
!ls -lh output/test_*.tsv

In [ ]:
# 3) Test unions (fast) + backup to Drive. End of Part A.
S = "code/business_entity_resolution/src"
!for C in India US France; do python3 $S/union_candidates.py --a output/test_${C}_name.tsv --b output/test_${C}_addr.tsv --k-per-file 40 --out output/test_union_${C}.tsv; cp output/test_union_${C}.tsv /content/drive/MyDrive/test_union_${C}.tsv; done
!ls -lh output/test_union_*.tsv

## Part A done — send back

1. Dry-run output (cell 1) — sent before cell 2.
2. `ls -lh` of `output/test_*.tsv` (cell 2 + 3 tail).

Then open **Part B**: `02b_colab_train_and_submit.ipynb` (train + inference +
submission). It restores Part A files from Drive if the runtime is fresh.